# Ledger: General 

This notebook shows the general ledger for the LLC




In [3]:
# Load bookkeeping services
import os
from ledger.LLC import LLC
from pathlib import Path
import datetime
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

top = os.path.join(Path.home(), 'GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group')
llc = LLC('WBGroupLLC',debug=True, top=top)  # debug='details'

llc:LLC init load _Profile
LLC llcProfile FN /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/llcProfile_WBGroupLLC.json
Profile loaded /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/llcProfile_WBGroupLLC.json
llc:LLC LLC Init Done


In [4]:
llc._Bank()

llcBank llcBank Init Done
llcBank llcBank Init Done
llcBank dwnLdCSV: importBankCSV csvBN: WBGroupLLC_WF_20251231.csv
llcBank CSV Loaded /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/BankStmts/WBGroupLLC_WF_20251231.csv
llcBank.wrangleLedger: llc assets:13
llcBank.wrangleLedger: llc owners:13
llcBank.wrangleLedger: llc customers:1
ledgerClassify ledgerClassify Init Done
llcBank.wrangleLedger: transList:54
llcBank.wrangleLedger: ********** Misc:7 Potential New, unClassified Ttansactions*****


In [5]:
llc.bk.df.groupby('Acct').Acct.count().loc['Acct.Cash.Misc'] 

np.int64(7)

In [6]:
begBal = llc._assets()._BegBal

# 
dtReport = datetime.datetime.now().strftime('%Y.%m.%d')

display(Markdown(f"### Profile - {dtReport}"))
display(Markdown(f"- **LLC Name: {llc.objName}**"))
display(Markdown(f"- **Year: {llc.yr}**\nAcct.Equity.Cash Beginning Balance: {begBal}"))

### Profile - 2026.04.03

- **LLC Name: WBGroupLLC**

- **Year: 2025**
Acct.Equity.Cash Beginning Balance: <bound method llcAssets._BegBal of <ledger.llcAssets.llcAssets object at 0x1115ef890>>

## General Ledger: Incomes/Expenses

In [7]:
# General Ledger - All accounts
llc._Bank()
glDF = llc.bk.df.groupby(['Acct']).amt.sum()
glDF.loc[f'Balance'] = glDF.sum()
print(glDF.to_string())

llcBank llcBank Init Done
llcBank llcBank Init Done
llcBank dwnLdCSV: importBankCSV csvBN: WBGroupLLC_WF_20251231.csv
llcBank CSV Loaded /Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/2025/BankStmts/WBGroupLLC_WF_20251231.csv
llcBank.wrangleLedger: llc assets:13
llcBank.wrangleLedger: llc owners:13
llcBank.wrangleLedger: llc customers:1
ledgerClassify ledgerClassify Init Done
llcBank.wrangleLedger: transList:54
llcBank.wrangleLedger: ********** Misc:7 Potential New, unClassified Ttansactions*****
Acct
Acct.Cash.Expense           -1766.92
Acct.Cash.Expense.Util      -1056.95
Acct.Cash.Income             4000.53
Acct.Cash.Income.Interest     400.00
Acct.Cash.Misc               5142.52
Balance                      6719.18


## Balance Sheet Ledger (Assets - Liabilities - Equity = 0)


In [8]:
# llcAssets.getBalanceSheeet
bsDF = llc.assets().getBalanceSheet()
bsDF

,Asset,Equity,Liability
Ledger,,,
Acct.Cash.Expense.Closing,662.25,,
Acct.Cash.Income.Interest,-50.0,,
Acct.Cash.Owner,1660.64,,
Acct.Cash.Owner,-5300.0,,
Acct.Equity.Depreciation.Accumulation,,15000.0,
Acct.Equity.Fixed.Tangible.InConstruction,,177.0,
Acct.Equity.Fixed.Tangible.InService,,213936.95,
Acct.Equity.LLC,,0.0,
Acct.Equity.Owner,,-231912.94,


In [9]:
t = bsDF.loc['Total']
t.Asset - t.Equity

-228.120000000009

In [10]:
# Investment Ledger
import pandas as pd
iDF = llc.bk.df[llc.bk.df.Acct.str.contains('Owner')].copy()
iDF = pd.concat([iDF, llc.bk.df[llc.bk.df.Acct.str.contains('Equity')].copy()])
#iDF = pd.DataFrame(iDF.groupby(['AcctSub','Acct']).amt.sum())
iDF = iDF[['Acct', 'dt', 'amt', 'TDesc', 'AcctSub']].reset_index(drop=True).copy()
iDF.set_index('Acct', drop=True, inplace=True)

# Compute balance : investment/purchases and bank
baliDF = round(iDF.amt.sum(),2)
balBk = round(float(llc.bk.df.amt.sum()),2)

iDF.loc['NetValue YE'] = ['12/31/2025',baliDF,'YE Balance','']
iDF.loc['Gain/Loss YE'] = ['12/31/2025',balBk-baliDF,'Income/Expenses','']
iDF.loc['Acct.Cash.Balance'] = ['12/31/2025',balBk,'YE Cash Balance','']

display(Markdown('<h2> Investment Ledger'))
display(iDF)

<h2> Investment Ledger

,dt,amt,TDesc,AcctSub
Acct,,,,
NetValue YE,12/31/2025,0.00,YE Balance,
Gain/Loss YE,12/31/2025,6719.18,Income/Expenses,
Acct.Cash.Balance,12/31/2025,6719.18,YE Cash Balance,


# Ledger - Income

In [11]:
# Investment Ledger
incDF = llc.bk.df[llc.bk.df.Acct.str.contains('.Income')].copy()
incDF = pd.DataFrame(incDF.groupby(['Acct','AcctSub','dt']).amt.sum())

# --- lookup customer name based on ID
cList = llc.customers()
custList = []
for (acct, ndx,dt) in incDF.index:
    for d in cList:
        if d['oID'] == ndx: custList.append(d['nm'])
        else: custList.append('na')
incDF['Customer'] = custList
incDF.loc[('Total','','')] = [incDF.amt.sum(),'']
display(Markdown('<h2> Income Ledger'))
display(incDF)

<h2> Income Ledger

amt  \
Acct                      AcctSub     dt                    
Acct.Cash.Income          c20251001-1 10/23/2025  1000.00   
                                      10/24/2025     0.53   
                                      11/12/2025  1500.00   
                                      12/01/2025  1500.00   
Acct.Cash.Income.Interest Bank        10/24/2025   400.00   
Total                                             4400.53   

                                                                              Customer  
Acct                      AcctSub     dt                                                
Acct.Cash.Income          c20251001-1 10/23/2025  [Nicola Rojas, Alejandro Villarreal]  
                                      10/24/2025  [Nicola Rojas, Alejandro Villarreal]  
                                      11/12/2025  [Nicola Rojas, Alejandro Villarreal]  
                                      12/01/2025  [Nicola Rojas, Alejandro Villarreal]  
Acct.Cash.Income.Interest Bank        10/24/2025                                    na  
Total

In [12]:
# Profit/Loss reconcile with Bank Stmt Balance
df = llc.bk.df.copy()
def _isProfitLoss(self, r):
    if 'Equity' in r.Acct : return 'Equity'
    if r.TransType == 'Exp' : return 'Expense'
    return 'Revenue'

# Classify Transactions into Exp, Rev or False (something else)
df['PnL'] = df.apply(lambda r: _isProfitLoss(llc, r), axis=1)

# Filter only Exp/Rev
plDF = pd.DataFrame(df[df.PnL != 'Equity'])

# Build table subcategories: P&L, Acct and ammount
pvt = pd.pivot_table(plDF, values='amt', index=['PnL', 'Acct'], aggfunc='sum')
pvt.loc[('Total','Profit(Loss)'),'amt'] = round(pvt.amt.sum(),2)
display(Markdown('<h2> Profit/Loss Ledger'))
display(pvt)

<h2> Profit/Loss Ledger

amt
PnL     Acct                                
Expense Acct.Cash.Expense           -1808.02
        Acct.Cash.Expense.Util      -1056.95
        Acct.Cash.Misc            -214114.48
Revenue Acct.Cash.Expense              41.10
        Acct.Cash.Income             4000.53
        Acct.Cash.Income.Interest     400.00
        Acct.Cash.Misc             219257.00
Total   Profit(Loss)                 6719.18

In [13]:
# Subtotal Expenses
expDF = pd.DataFrame(plDF[plDF.PnL == 'Expense'][['Acct','amt']].groupby('Acct').amt.sum())
balExp = round(expDF.amt.sum(),2)
expDF.loc['Subtotal'] = balExp
display(Markdown('<h2> Expense Ledger'))
display(expDF)

<h2> Expense Ledger

,amt
Acct,
Acct.Cash.Expense,-1808.02
Acct.Cash.Expense.Util,-1056.95
Acct.Cash.Misc,-214114.48
Subtotal,-216979.45


In [14]:
# Subtotal Expenses
revDF = pd.DataFrame(plDF[plDF.PnL == 'Revenue'][['Acct','amt']].groupby('Acct').amt.sum())
balRev = round(revDF.amt.sum(),2)
revDF.loc['Subtotal'] = balRev
display(Markdown('<h2> Revenue: : YE Summary per Account'))
display(revDF)

<h2> Revenue: : YE Summary per Account

,amt
Acct,
Acct.Cash.Expense,41.10
Acct.Cash.Income,4000.53
Acct.Cash.Income.Interest,400.00
Acct.Cash.Misc,219257.00
Subtotal,223698.63


In [15]:
# YE Summary : Gains/Loss
display(Markdown("<h2> Revenue: : YE Summary: Gains/Loss"))
pd.DataFrame([['Expense', 'Gains', 'Total Operating Gain/Loss'],
               [balExp, balRev, balRev+balExp]], index=['YE Summary','Amt']).transpose()

<h2> Revenue: : YE Summary: Gains/Loss

,YE Summary,Amt
0,Expense,-216979.45
1,Gains,223698.63
2,Total Operating Gain/Loss,6719.18


In [16]:
# Expense Ledger
df3 = llc.bk.df
display(Markdown("<h2> Expense Ledger - details"))
df3 = df3[df3.TransType == 'Exp']

pd.DataFrame(df3.groupby(['Acct','AcctSub']).amt.sum())

<h2> Expense Ledger - details

amt
Acct                   AcctSub               
Acct.Cash.Expense      Maintenance    -135.00
                       amazon         -235.49
                       h-e-b          -173.68
                       harbor          -60.58
                       hays            -32.00
                       kings           -25.94
                       laird          -487.13
                       lowe's          -27.04
                       lowes           -31.86
                       sp              -74.57
                       sq              -57.31
                       texas           -90.90
                       wal-mart       -140.73
                       wimberley      -235.79
Acct.Cash.Expense.Util Elec           -571.15
                       Ins_Home       -135.80
                       Util            -50.00
                       Water          -300.00
Acct.Cash.Misc         Misc        -214114.48

In [17]:
# WORK IN PROGRESS - Classify  Expenses
expDF = llc.bk.df[llc.bk.df.Acct.str.contains('Expense')].copy()
df2 = pd.DataFrame(expDF.groupby('AcctSub').amt.sum())
balExp = df2.sum()
df2.loc['Total'] = [balExp]
display(df2)
expDF[['dt', 'amt', 'Acct', 'AcctSub', 'TDesc']]

,amt
AcctSub,
Elec,-571.15
Ins_Home,-135.8
Maintenance,-135.0
Util,-50.0
Water,-300.0
amazon,-235.49
h-e-b,-173.68
harbor,-60.58
hays,-32.0


,dt,amt,Acct,AcctSub,TDesc
2,12/26/2025,-135.80,Acct.Cash.Expense.Util,Ins_Home,Pay Monthly Util
3,12/18/2025,-50.00,Acct.Cash.Expense.Util,Water,Pay Monthly Util
4,12/10/2025,-129.16,Acct.Cash.Expense.Util,Elec,Pay Monthly Util
6,11/18/2025,-50.00,Acct.Cash.Expense.Util,Water,Pay Monthly Util
7,11/17/2025,-2.00,Acct.Cash.Expense,hays,Expense: 11/15 hays co tx wimber fort worth tx...
8,11/17/2025,-30.00,Acct.Cash.Expense,hays,Expense: 11/15 hays co tx wimber san marcos tx...
9,11/17/2025,-19.47,Acct.Cash.Expense,amazon,Expense: 11/15 amazon mktpl*b80w8 amzn.com/bil...
10,11/17/2025,-32.42,Acct.Cash.Expense,sp,Expense: 11/14 sp growers solutio growerssolut...
12,11/10/2025,-119.49,Acct.Cash.Expense.Util,Elec,Pay Monthly Util
13,11/07/2025,-42.15,Acct.Cash.Expense,sp,Expense: 11/06 sp growers solutio growerssolut...


In [18]:
# Miscellenous / Unclassified Transactions
display(Markdown("<h2> Ledger - Misc Accounts"))

llc.bk.df[llc.bk.df.Acct.str.contains('Misc')][['dt', 'amt', 'Acct', 'AcctSub', 'desc']]


<h2> Ledger - Misc Accounts

,dt,amt,Acct,AcctSub,desc
0,12/29/2025,-177.00,Acct.Cash.Misc,Misc,Cash eWithdrawal in Branch 12/29/2025 13:47 PM...
1,12/29/2025,177.00,Acct.Cash.Misc,Misc,eDeposit in Branch 12/29/25 03:48:15 PM 14650 ...
16,10/24/2025,-0.53,Acct.Cash.Misc,Misc,TRUIST ACCTVERIFY 251024 15280212831 ALEJANDRO...
45,09/29/2025,30.00,Acct.Cash.Misc,Misc,MOBILE DEPOSIT : REF NUMBER :409290718471
51,08/26/2025,-213936.95,Acct.Cash.Misc,Misc,WITHDRAWAL MADE IN A BRANCH/STORE
52,08/20/2025,50.00,Acct.Cash.Misc,Misc,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...
53,08/20/2025,219000.00,Acct.Cash.Misc,Misc,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...


In [19]:
aObj = llc.assets()
aObj.fetch()
r = llc.bk.df.iloc[51]
k = aObj._key(r)
print("---", k, "<<<")
aObj._matchBk(r)
i = aObj.kDict[k[0]]
aObj.df.iloc[i]

--- ('08/26/2025_-213936.95', 51) <<<


aID                                                e20250826-805HMD
addr                             805 High Mesa Dr, Wimberley, 78676
amt                                                       213936.95
dt                                                       2025.08.26
stakeholderPct                                       {'LLC': 100.0}
acct                                                Acct.Cash.Owner
aType                                                        Credit
propNm                                                H_805HighMesa
propRef                                            e20250826-805HMD
desc                               Purchase Property: 805 High Mesa
Ledger                         Acct.Equity.Fixed.Tangible.InService
kwList                                                           []
amtType                                                       debit
basis             {'salePrice': 220000, 'closingCost': 662.0, 'p...
improvements      [{'nm': 'plumbing', 'dt': '202

In [20]:
# Owners DB
display(Markdown("<h2> llcOwners DB"))

pd.DataFrame(llc.owners())

<h2> llcOwners DB

,oID,nm,addr,status,memType,pct,kw
0,o20250801-1,[Francis X Rojas],"177 Kingsway Dr, Wimberley, 78676",Manager,active,0.96,[WT FED#02M03 NATIONAL FINANCIAL]
1,o20250801-2,[Alexandra Rojas],TBD,non_active member,passive,0.02,[]
2,o20250801-3,[Nicola Rojas],"805 High Mesa Dr, Wimberley, 78676",non-active member,passive,0.02,[]


In [21]:
# Customer DB
display(Markdown("<h2> llcCustomers DB"))
pd.DataFrame(llc.customers())

<h2> llcCustomers DB

,oID,nm,addr,rent,start,kwList
0,c20251001-1,"[Nicola Rojas, Alejandro Villarreal]","805 High Mesa Dr, Wimberley, 78676",1500.0,2025.10.01,[]


In [22]:
# Asset DB
from ledger.llcAssets import llcAssets

a = llcAssets(llc)
a.fetch()


display(Markdown("<h2> llcAsset DB"))
a.df

<h2> llcAsset DB

,aID,addr,amt,dt,stakeholderPct,acct,aType,propNm,propRef,desc,Ledger,kwList,amtType,basis,improvements,pct
0,a20250820-Cash1,"177 Kingsway Dr, Wimberley, 786767",0.00,2025.08.20,{'o20250801_1': 100.0},Acct.Cash.Balance,Debit,Cash_LLC,a20250820-Cash1,YR.2025.Beg.Cash,Acct.Equity.LLC,[],NaN,NaN,NaN,NaN
1,a20250826-Cash1,"177 Kingsway Dr, Wimberley, 786767",219000.00,2025.08.20,{'o20250801_1': 100.0},Acct.Cash.Owner,Debit,H_805HighMesa,e20250826-805HMD,Initial Investment by member,Acct.Equity.Owner,[],NaN,NaN,NaN,NaN
2,a20250826-Cash2,"177 Kingsway Dr, Wimberley, 786767",50.00,2025.08.20,{'o20250801_1': 100.0},Acct.Cash.Income.Bank,Debit,H_805HighMesa,e20250826-805HMD,Open Bank Acct Investment,Acct.Cash.Income.Interest,[],NaN,NaN,NaN,NaN
3,e20250826-805HMD,"805 High Mesa Dr, Wimberley, 78676",213936.95,2025.08.26,{'LLC': 100.0},Acct.Cash.Owner,Credit,H_805HighMesa,e20250826-805HMD,Purchase Property: 805 High Mesa,Acct.Equity.Fixed.Tangible.InService,[],debit,"{'salePrice': 220000, 'closingCost': 662.0, 'p...","[{'nm': 'plumbing', 'dt': '2025.04', 'amt': 86...","{'Land': 0.36, 'Bld': 0.64}"
4,a20250826-Cash4,"177 Kingsway Dr, Wimberley, 786767",662.25,2025.08.20,{'o20250801_1': 100.0},Acct.Equity.Fixed.Tangible.InService,Credit,H_805HighMesa,e20250826-805HMD,Closing Cost Expense,Acct.Cash.Expense.Closing,[],NaN,NaN,NaN,NaN
5,a20250826-Cash5,"177 Kingsway Dr, Wimberley, 786767",1660.64,2025.08.20,{'o20250801_1': 100.0},Acct.Equity.Fixed.Tangible.InService,Debit,H_805HighMesa,e20250826-805HMD,"PrePaid Prop Tax, seller paid - assign to valu...",Acct.Equity.Owner,[],NaN,NaN,NaN,NaN
6,l20250826-Cash6,"177 Kingsway Dr, Wimberley, 786767",-1660.64,2025.08.20,{'o20250801_1': 100.0},Acct.Equity.Owner,Credit,H_805HighMesa,e20250826-805HMD,PrePaid Prop Tax - realize cash from value of ...,Acct.Cash.Owner,[],NaN,NaN,NaN,NaN
7,d20250826-Dep1,"177 Kingsway Dr, Wimberley, 786767",15000.00,2025.08.20,{'o20250801_1': 100.0},Acct.Equity.Expense.Depreciation,Credit,H_805HighMesa,e20250826-805HMD,"Improvements @ Purchase, lower basis",Acct.Equity.Depreciation.Accumulation,[],NaN,NaN,NaN,NaN
8,a20250826-Cash7,"177 Kingsway Dr, Wimberley, 786767",5775.30,2025.08.20,{'o20250801_1': 100.0},Acct.Equity.Fixed.Tangible.InService,Debit,H_805HighMesa,e20250826-805HMD,"Bal of Cash @ closing, see Buyers Closing Stat...",Acct.Equity.Owner,[],NaN,NaN,NaN,NaN
9,a20250826-Escrow,"177 Kingsway Dr, Wimberley, 786767",5300.00,2025.08.20,{'o20250801_1': 100.0},Acct.Cash.Owner,Debit,H_805HighMesa,e20250826-805HMD,"Escrow Invested by member,see Buyers Closing S...",Acct.Equity.Owner,[],NaN,NaN,NaN,NaN


# Accounting-Bookkeeping 101

## accounting ledgers: 

| Ledger | Description |
| :---- | :----: |
|General Ledger | master document; record all transactions; includes all accounts related to a company's assets, liabilities, equity, revenue, and expenses.|
|Sales Ledger | |
|Purchase Ledger |  

## Key Principles (pyApps)

- utilizing pythn accounting libraries
- double-entry bookkeeping, or by
- building a custom application (general ledger for rental LLC)
- a web framework / command-line tool
- read bank statements (often from CSV files)
- perform transactions and reporting. 

### Beancount: 
A Python package for double-entry accounting. You can use its command-line tools to manage financial transactions written in a plain-text file, making it suitable for tracking an LLC's finances and maintaining a Git-based audit trail.
### Blnk Finance: 
An open-source, developer-focused toolkit that includes a double-entry ledger for managing balances and transactions. It offers features like balance monitoring, reconciliation, and identity management, and is designed to help you build fintech products.
### Django Ledger: 
An open-source accounting and financial ledger system built on the Django framework. It's a good option if you need to integrate robust accounting features directly into a Python web application. 

### Github Py Ledgers

1. [**>Confidential Ledger Azure**](https://learn.microsoft.com/en-us/python/api/overview/azure/confidentialledger-readme?view=azure-python#key-concepts)
    - good reference
1. [**> Ledger Visualization**](https://wilw.dev/blog/2022/04/24/ledger-python-visualisation/) : pure text (csv) files, uses vis.py 

## Ledger for Taxes

- bare minimum records
- accounting ledger for an LLC to file taxes
- clear, accurate summary
  - all business income (gross receipts)
  - expenses.
- no specific bookkeeping method
- method used must clearly reflect your income and expenses. 

## key information required for each transaction includes:

- Amount of the transaction.
- Date of the transaction.
- Description of the item purchased or service received.
- Business purpose (why the expense was necessary).
- Source of income or the payee for expenses. 
- Essential Records and Documentation

## Documentation
The IRS requires that you maintain documentation to support the figures reported on your tax return. 

- Well-organized,
- detailed records make tax preparation easier
- help you maximize deductions
- ensuring you have the necessary documentation in case of an IRS audit.
- Records: keep at least three years from the date you filed the return. 


|Record Type 	|Description	|Supporting Documents to Keep|
| ---- | ---- | ---- |
|Income	|All gross receipts from business operations.	|Invoices sent to clients, cash register tapes, deposit slips, bank account deposits, and Forms 1099-NEC received.
|Expenses	|All costs incurred to carry on your business.|	Canceled checks, credit card statements, invoices for purchases, and receipts.|
|Assets & Depreciation|	Records for property like equipment or vehicles used in the business that last more than a year.	|Date and method of acquisition, purchase price, cost of improvements, deductions taken for depreciation, business use, and details of disposition/sale.|
|Payroll (if applicable)|	Records related to employees (not owners, typically).|	Annual W-2s, quarterly and annual payroll tax returns, and complete pay records/time sheets for each employee.|

### Multi-member LLC: Filing Requirements by LLC Structure
- Your LLC's tax classification determines which forms you will need to file, which impacts how you report your income and expenses. 
- Multi-member LLC: The default is to be taxed as a partnership. The LLC files an informational Form 1065 and provides a Schedule K-1 to each member, who then reports their share of profit or loss on their personal Form 1040 using Schedule E.
- LLC taxed as a Corporation (S corp or C corp): If you elect this status, you will file Form 1120 (C corp) or Form 1120-S (S corp). 


# Accounting: Best Practices

## Rental Accounting

|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Acct.Cash.Income     | ##.## |       | yy.mm.dd | CustID  | record the cash received |
|Acct.Rent.Receivable | ##.## |       | yy.mm.dd | CustID  | record amt owed by the tenant|
|Income.Rent.Revenue  |       | ##.## | yy.mm.dd | CustID  | record total income earned, period |

## Initial investment of cash
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Acct.Cash.Investment |       | ##.## | yy.mm.dd |           | record cash investment|
|LLC.Equity.Member    | ##.## |       | yy.mm.dd | memberID  | LLC equity, member % |

## Company purchasing an investment
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|LLC.Prop.Investment  | ##.## |        | yy.mm.dd | propID | record value of purchase/investment|
|Acct.Cash.Purchase   |       | ##.##  | yy.mm.dd | propID | Cash account reduced |

## Expense: Paid by Cash / Debit Card
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Acct.Cash.Expense    |       | ##.##  | yy.mm.dd | expCls | record expense details|
|Acct.Expense.<expCls>| ##.## |        | yy.mm.dd | expCls | record expense details|


## Expense: Paid by accured (credit card/loan-agreement/late payment)
|Account              |	Debit | Credit| Date | SubAcct | Description |
| ------------------- | :----- | :----: | :----- | :----: | :----- |
|Liab.Payable.Expense |       | ##.## | yy.mm.dd | expCls | record expense details |
|Liab.Expense.<expCls>| ##.## |       | yy.mm.dd | expCls | record expense details|
| | | | | | future |
|Acct.Cash.Expense    |       | ##.## | yy.mm.dd | expCls | record expense details|
|Acct.Expense.<expCls>| ##.## |       | yy.mm.dd | expCls | record expense details|


Example: Recording rent expense Transaction: A company pays \(\$1,000\) for monthly rent.Debit: The Rent Expense account is debited for \(\$1,000\).Credit: The Cash account is credited for \(\$1,000\). 

Note 3Example: Recording an accrued expense Transaction: At the end of the month, a company has an accrued electricity expense of \(\$300\) that has not yet been paid.Debit: The Electricity Expense account is debited for \(\$300\).Credit: The Accrued Expenses (or Electricity Expense Payable) account is credited for \(\$300\) to create the liability. 


## NOTE 1: Subsequent changes to the investment
- accounting for an investment becomes more complex, 
- value of investment fair value method or equity method,
- depending on the type of investment.

## NOTE 2: Earnings from investments
- recognized in the `income statement`
- dividends received are often treated as either operating or investment inflows
- `cash flow statement`.

## NOTE 3: accrual method
- expenses are recorded when they are incurred, not when they are paid.
- Example: Recording an accrued expense Transaction:
    - end of the month, not paid elect expense 
    - a company has an accrued electricity expense of $300
    - Debited $300 :  Expense.Utility.Elec
    - Credit $300 : Liab.Expense.Utility
    - The Accrued Expenses (or Electricity Expense Payable) - create the liability.
    - 
## NOTE 4: Investments on form 1065
- An LLC taxed as a partnership records owner investments (`capital contributions`) on IRS Form 1065,
- primarily on `Schedule L (Balance Sheets per Books)` and
- Schedule M-2 (Analysis of Partners' Capital Accounts).
- Investments are considered `capital contributions` rather than income
- Capital contributions are tracked to determine the partners' basis in the LLC